Raw Dataset Mergring to create one Dataset

This project focuses on data preprocessing and feature engineering using Formula 1 datasets.
The goal is to merge multiple datasets, clean the data, and create useful features for analysis or machine learning.

In [ ]:
import pandas as pd

# Load datasets
drivers = pd.read_csv('/content/drivers.csv')
constructors = pd.read_csv('/content/constructors.csv')
races = pd.read_csv('/content/races.csv')
results = pd.read_csv('/content/results.csv')
qualifying = pd.read_csv('/content/qualifying.csv')

# Merge safely step by step
df = results.merge(races[['raceId','year']], on='raceId', how='left')
df = df.merge(qualifying[['raceId','driverId','q1','q2','q3']], on=['raceId','driverId'], how='left')

# Select important columns
df = df[['raceId','year','driverId','constructorId','grid','positionOrder','q1','q2','q3']]

# Clean
df = df.replace('\\N', None)
df = df.dropna()

df['final_position'] = pd.to_numeric(df['positionOrder'])
df['grid'] = pd.to_numeric(df['grid'])

# Feature
df['top3'] = df['final_position'].apply(lambda x: 1 if x <= 3 else 0)

df = df.drop(columns=['positionOrder'])

# Save
df.to_csv("f1_clean_dataset_final.csv", index=False)

In [ ]:
import pandas as pd

df = pd.read_csv('f1_clean_dataset_final.csv')

df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df['top3'].value_counts()

In [ ]:
import matplotlib.pyplot as plt


In [ ]:
plt.scatter(df['grid'], df['final_position'])
plt.xlabel("Grid Position")
plt.ylabel("Final Position")
plt.title("Grid vs Final Position")
plt.show()

In [ ]:
df['top3'].value_counts().plot(kind='bar')
plt.title("Top 3 vs Others")
plt.show()

In [ ]:
df.groupby('year')['final_position'].mean().plot()
plt.title("Average Position Over Years")
plt.show()

In [ ]:
df[['grid','final_position','top3']].corr()

In [ ]:
import seaborn as sns

sns.heatmap(df[['grid','final_position','top3']].corr(), annot=True)
plt.show()

In [ ]:
def convert_time_to_seconds(time):
    try:
        minutes, seconds = time.split(':')
        return float(minutes) * 60 + float(seconds)
    except:
        return None

df['q1_sec'] = df['q1'].apply(convert_time_to_seconds)
df['q2_sec'] = df['q2'].apply(convert_time_to_seconds)
df['q3_sec'] = df['q3'].apply(convert_time_to_seconds)

In [ ]:
df['best_qualifying'] = df[['q1_sec','q2_sec','q3_sec']].min(axis=1)

In [ ]:
df['qualifying_rank'] = df['best_qualifying'].rank()

In [ ]:
df['grid_advantage'] = df['grid'] - df['final_position']

In [ ]:
df = df.drop(columns=['q1','q2','q3'])

In [ ]:
df.head()

In [ ]:
from sklearn.model_selection import train_test_split

# Features
X = df[['grid','best_qualifying','qualifying_rank']]

# Targets
y_reg = df['final_position']   # Regression
y_clf = df['top3']            # Classification

# Split data
X_train, X_test, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, random_state=42)
_, _, y_train_clf, y_test_clf = train_test_split(X, y_clf, test_size=0.2, random_state=42)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

lr = LinearRegression()
lr.fit(X_train, y_train_reg)

pred_lr = lr.predict(X_test)

print("MAE:", mean_absolute_error(y_test_reg, pred_lr))
print("R2 Score:", r2_score(y_test_reg, pred_lr))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor()
rf.fit(X_train, y_train_reg)

pred_rf = rf.predict(X_test)

print("MAE:", mean_absolute_error(y_test_reg, pred_rf))
print("R2 Score:", r2_score(y_test_reg, pred_rf))

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

log = LogisticRegression(max_iter=1000, class_weight='balanced')
log.fit(X_train, y_train_clf)

pred_log = log.predict(X_test)

print("Accuracy:", accuracy_score(y_test_clf, pred_log))
print(classification_report(y_test_clf, pred_log))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(class_weight='balanced')
rf_clf.fit(X_train, y_train_clf)

pred_rf_clf = rf_clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test_clf, pred_rf_clf))
print(classification_report(y_test_clf, pred_rf_clf))

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt_clf = DecisionTreeClassifier()
dt_clf.fit(X_train, y_train_clf)

pred_dt_clf = dt_clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test_clf, pred_dt_clf))
print(classification_report(y_test_clf, pred_dt_clf))

In [ ]:
from sklearn.model_selection import cross_val_score

rf = RandomForestRegressor()

scores = cross_val_score(rf, X, y_reg, cv=5, scoring='r2')

print("Cross Validation R2 Scores:", scores)
print("Average R2:", scores.mean())

In [ ]:
from sklearn.model_selection import cross_val_score

rf = RandomForestRegressor()

scores = cross_val_score(rf, X, y_reg, cv=5, scoring='r2')

print("Cross Validation R2 Scores:", scores)
print("Average R2:", scores.mean())

In [ ]:
model = RandomForestRegressor()
model.fit(X_train, y_train_reg)

import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
importance.sort_values().plot(kind='barh')

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(class_weight='balanced')

scores = cross_val_score(rf_clf, X, y_clf, cv=5, scoring='f1')

print("F1 Scores:", scores)
print("Average F1:", scores.mean())

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_test_clf, pred_rf_clf)